In [1]:
!pip install -q git+https://github.com/openai/whisper.git
!pip install -q yt-dlp  # Switched from pytube to yt-dlp for reliability
!pip install -q transformers[torch]
!pip install -q accelerate
!pip install -q langchain-community
!pip install -q langchain-huggingface
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q python-docx
!pip install -q reportlab
!pip install -q gradio
!pip install -q openai-whisper
!apt-get install -qq ffmpeg


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.0/176.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 467.1/467.1 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
langchain 0.3.27 requires langchain-core<1.

In [2]:
import os
import warnings
warnings.filterwarnings('ignore')

In [3]:
import whisper
import yt_dlp

In [4]:
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFacePipeline
from docx import Document
from docx.shared import Inches, Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
import gradio as gr
from datetime import datetime
import re
import urllib.parse
print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


In [7]:
class AudioExtractor:
    def __init__(self):
        self.model = None
    def load_whisper_model(self, model_size="base"):
        """Load Whisper model for speech-to-text"""
        print(f"📥 Loading Whisper {model_size} model...")
        self.model = whisper.load_model(model_size)
        print("✅ Whisper model loaded!")
        return self.model
    def download_audio(self, youtube_url):
        """Download audio from YouTube video using yt-dlp"""
        try:
            youtube_url = self.clean_url(youtube_url)  # Clean URL before use
            print(f"🎥 Downloading audio from: {youtube_url}")
            # yt-dlp options for audio download
            ydl_opts = {
                'format': 'bestaudio/best',  # Best audio quality
                'outtmpl': 'temp_audio.%(ext)s',  # Output filename
                'postprocessors': [{
                    'key': 'FFmpegExtractAudio',
                    'preferredcodec': 'mp3',  # Convert to MP3
                    'preferredquality': '192',
                }],
                'quiet': True,  # Suppress verbose output
                'no_warnings': True,
            }
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                info = ydl.extract_info(youtube_url, download=True)
                video_title = info.get('title', 'Unknown Title')
                output_file = ydl.prepare_filename(info).replace('.webm', '.mp3').replace('.m4a', '.mp3')  # Ensure MP3 extension
            print(f"✅ Audio downloaded: {video_title}")
            return output_file, video_title
        except Exception as e:
            print(f"❌ Error downloading audio: {str(e)}")
            return None, None
    def transcribe_audio(self, audio_file):
        """Transcribe audio to text using Whisper"""
        try:
            print("🔊 Transcribing audio...")
            result = self.model.transcribe(audio_file, verbose=False)
            transcript = result["text"]
            print(f"✅ Transcription complete! Length: {len(transcript)} characters")
            return transcript
        except Exception as e:
            print(f"❌ Error transcribing: {str(e)}")
            return None
    def clean_url(self, url):
        """Remove unwanted characters or URL encoding"""
        url = url.strip().strip('"').strip("'")
        url = urllib.parse.unquote(url)  # Removes %27 etc.
        return url

In [10]:
class TextSummarizer:
    def __init__(self):
        self.summarizer = None
        self.tokenizer = None

    def load_model(self, model_name="facebook/bart-large-cnn"):
        """Load summarization model"""
        print(f"📥 Loading summarization model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
        self.summarizer = pipeline(
            "summarization",
            model=model,
            tokenizer=self.tokenizer,
            device=0 if torch.cuda.is_available() else -1
        )
        print("✅ Summarization model loaded!")

    def chunk_text(self, text, max_chunk_size=1000):
        """Split text into manageable chunks"""
        sentences = re.split(r'(?<=[.!?])\s+', text)
        chunks = []
        current_chunk = ""

        for sentence in sentences:
            if len(current_chunk) + len(sentence) < max_chunk_size:
                current_chunk += sentence + " "
            else:
                if current_chunk:
                    chunks.append(current_chunk.strip())
                current_chunk = sentence + " "

        if current_chunk:
            chunks.append(current_chunk.strip())

        return chunks

    def summarize_text(self, text):
        """Summarize text using the loaded model"""
        if not self.summarizer:
            print("❌ Summarization model not loaded. Please call load_model() first.")
            return None

        try:
            print("✨ Summarizing text...")
            chunks = self.chunk_text(text)
            summaries = []

            for i, chunk in enumerate(chunks):
                print(f"   Summarizing chunk {i+1}/{len(chunks)}...")
                summary = self.summarizer(
                    chunk,
                    max_length=150,
                    min_length=50,
                    do_sample=False
                )
                summaries.append(summary[0]['summary_text'])

            combined_summary = " ".join(summaries)

            # If combined is still too long, summarize again
            if len(combined_summary.split()) > 500:
                print("   Creating final summary...")
                final_summary = self.summarizer(
                    combined_summary,
                    max_length=300,
                    min_length=100,
                    do_sample=False
                )
                combined_summary = final_summary[0]['summary_text']

            print("✅ Summary generated!")
            return combined_summary

        except Exception as e:
            print(f"❌ Error summarizing: {str(e)}")
            return None


    def extract_key_points(self, text):
        """Extract key points from text"""
        sentences = re.split(r'(?<=[.!?])\s+', text)
        # Take important sentences (you can enhance this with NLP techniques)
        key_points = sentences[:min(10, len(sentences))]
        return key_points

In [11]:
class StudyGuideGenerator:
    def __init__(self):
        pass

    def create_study_guide(self, video_title, transcript, summary, key_points, output_path="study_guide.docx"):
        """Generate a formatted study guide document"""
        try:
            print("📄 Creating study guide...")

            doc = Document()

            # Title
            title = doc.add_heading('Synapse Study Guide', 0)
            title.alignment = WD_ALIGN_PARAGRAPH.CENTER

            # Video Title
            video_heading = doc.add_heading(video_title, 1)
            video_heading.alignment = WD_ALIGN_PARAGRAPH.CENTER

            # Date
            date_para = doc.add_paragraph(f"Generated on: {datetime.now().strftime('%B %d, %Y')}")
            date_para.alignment = WD_ALIGN_PARAGRAPH.CENTER

            doc.add_paragraph()

            doc.add_heading('📋 Executive Summary', 2)
            doc.add_paragraph(summary)

            doc.add_paragraph()

            # Key Points Section
            doc.add_heading('🔑 Key Points', 2)
            for i, point in enumerate(key_points, 1):
                doc.add_paragraph(point, style='List Number')

            doc.add_paragraph()

            # Full Transcript Section
            doc.add_heading('📝 Full Transcript', 2)
            transcript_para = doc.add_paragraph(transcript[:2000] + "..." if len(transcript) > 2000 else transcript)

            # Save document
            doc.save(output_path)
            print(f"✅ Study guide saved: {output_path}")
            return output_path

        except Exception as e:
            print(f"❌ Error creating study guide: {str(e)}")
            return None

In [12]:
class StudyChatbot:
    def __init__(self):
        self.vector_store = None
        self.text_chunks = []
        self.qa_model = None
        self.qa_tokenizer = None

    def setup_qa_system(self, text):
        """Setup simplified Q&A system"""
        try:
            print("🤖 Setting up chatbot...")

            # Split text into chunks
            text_splitter = RecursiveCharacterTextSplitter(
                chunk_size=500,
                chunk_overlap=50
            )
            self.text_chunks = text_splitter.split_text(text)
            print(f"   Created {len(self.text_chunks)} text chunks")

            # Create embeddings
            print("   Creating embeddings...")
            embeddings = HuggingFaceEmbeddings(
                model_name="sentence-transformers/all-MiniLM-L6-v2"
            )

            # Create vector store
            print("   Building vector store...")
            self.vector_store = FAISS.from_texts(self.text_chunks, embeddings)
            # Setup QA model
            print("   Loading QA model...")
            model_name = "google/flan-t5-base"
            self.qa_tokenizer = AutoTokenizer.from_pretrained(model_name)
            self.qa_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

            if torch.cuda.is_available():
                self.qa_model = self.qa_model.to('cuda')

            print("✅ Chatbot ready!")
            return True

        except Exception as e:
            print(f"❌ Error setting up chatbot: {str(e)}")
            return False

    def ask_question(self, question):
        """Answer a question based on the study material"""
        try:
            if self.vector_store is None:
                return "❌ Chatbot not initialized. Please process a video first."

            # Retrieve relevant chunks
            docs = self.vector_store.similarity_search(question, k=3)
            context = "\n".join([doc.page_content for doc in docs])

            # Create prompt
            prompt = f"Context: {context}\n\nQuestion: {question}\n\nAnswer:"
            # Generate answer
            inputs = self.qa_tokenizer(
                prompt,
                return_tensors="pt",
                max_length=512,
                truncation=True
            )

            if torch.cuda.is_available():
                inputs = {k: v.to('cuda') for k, v in inputs.items()}

            outputs = self.qa_model.generate(
                **inputs,
                max_length=150,
                num_beams=4,
                early_stopping=True
            )

            answer = self.qa_tokenizer.decode(outputs[0], skip_special_tokens=True)

            return answer

        except Exception as e:
            return f"❌ Error: {str(e)}"


In [ ]:
class SynapsePipeline:
    def __init__(self):
        self.audio_extractor = AudioExtractor()
        self.summarizer = TextSummarizer()
        self.guide_generator = StudyGuideGenerator()
        self.chatbot = StudyChatbot()

        self.current_transcript = None
        self.current_summary = None
        self.current_title = None

    def initialize_models(self):
        """Load all required models"""
        print("🚀 Initializing Synapse...")
        self.audio_extractor.load_whisper_model("base")
        self.summarizer.load_model("facebook/bart-large-cnn")
        print("✅ All models loaded!")

    def process_video(self, youtube_url):
        """Main pipeline to process YouTube video"""
        try:
            # Step 1: Download and transcribe
            audio_file, video_title = self.audio_extractor.download_audio(youtube_url)
            if not audio_file:
                return "❌ Failed to download video", None, None

            transcript = self.audio_extractor.transcribe_audio(audio_file)
            if not transcript:
                return "❌ Failed to transcribe audio", None, None

            # Step 2: Summarize
            summary = self.summarizer.summarize_text(transcript)
            if not summary:
                return "❌ Failed to generate summary", None, None

            key_points = self.summarizer.extract_key_points(summary)

            # Step 3: Create study guide
            guide_path = self.guide_generator.create_study_guide(
                video_title, transcript, summary, key_points
            )

            # Step 4: Setup chatbot
            self.chatbot.setup_qa_system(transcript)

            # Store current data
            self.current_transcript = transcript
            self.current_summary = summary
            self.current_title = video_title

            # Cleanup
            if os.path.exists(audio_file):
                os.remove(audio_file)

            return summary, guide_path, "✅ Video processed successfully! You can now ask questions."

        except Exception as e:
            return f"❌ Error: {str(e)}", None, None

    def chat(self, question):
        """Handle chat interactions"""
        if not self.current_transcript:
            return "⚠️ Please process a video first before asking questions."

        return self.chatbot.ask_question(question)

def create_gradio_interface():
    """Create beautiful Gradio interface"""

    # Initialize pipeline
    pipeline = SynapsePipeline()
    pipeline.initialize_models()

    def process_wrapper(url):
        summary, guide_path, status = pipeline.process_video(url)
        return summary, guide_path, status

    def chat_wrapper(message, history):
        response = pipeline.chat(message)
        return response

    # Create interface
    with gr.Blocks(theme=gr.themes.Soft(), title="Synapse - AI Study Assistant") as demo:
        gr.Markdown("""
        # 🚀 Synapse - AI-Powered Study Assistant
        ### Transform YouTube lectures into study guides and chat with the content!
        """)

        with gr.Tab("📥 Video Processing"):
            with gr.Row():
                url_input = gr.Textbox(
                    label="YouTube URL",
                    placeholder="Enter YouTube video URL...",
                    scale=4
                )
                process_btn = gr.Button("🔄 Process Video", variant="primary", scale=1)

            status_output = gr.Textbox(label="Status", lines=2)

            with gr.Row():
                summary_output = gr.Textbox(
                    label="📋 Summary",
                    lines=10,
                    placeholder="Summary will appear here..."
                )
                guide_output = gr.File(label="📄 Download Study Guide")


            process_btn.click(
                fn=process_wrapper,
                inputs=[url_input],
                outputs=[summary_output, guide_output, status_output]
            )

        with gr.Tab("💬 Chat with Content"):
            gr.Markdown("### Ask questions about the video content")
            chatbot_interface = gr.ChatInterface(
                fn=chat_wrapper,
                examples=[
                    "What are the main points?",
                    "Can you explain the key concepts?",
                    "What are the important takeaways?",
                    "Summarize the content in 3 sentences"
                ],
                title="Study Assistant Chat"
            )

        gr.Markdown("""
        ---
        **How to use:**
        1. Paste a YouTube URL in the Video Processing tab
        2. Click "Process Video" and wait for processing to complete
        3. Download your study guide or go to Chat tab to ask questions!

        **Note:** Processing may take 3-5 minutes depending on video length.
        """)

    return demo
"""
Launch Cell - Run this to start the application
"""
# Create and launch interface
print("🎨 Creating Gradio interface...")
demo = create_gradio_interface()
demo.launch(share=False, debug=False)

🎨 Creating Gradio interface...
🚀 Initializing Synapse...
📥 Loading Whisper base model...


100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 104MiB/s]


✅ Whisper model loaded!
📥 Loading summarization model: facebook/bart-large-cnn


config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

   Summarizing chunk 14/14...
   Creating final summary...
✅ Summary generated!
📄 Creating study guide...
✅ Study guide saved: study_guide.docx
🤖 Setting up chatbot...
   Created 29 text chunks
   Creating embeddings...
   Building vector store...
   Loading QA model...
✅ Chatbot ready!
🎥 Downloading audio from: https://www.youtube.com/watch?v=62DxELjuRec&pp=ygUcIDUgbWlucyBoaXN0b3J5IHZpZGVvcyBlc3NheQ==
✅ Audio downloaded: The Great Depression - 5 Minute History Lesson
🔊 Transcribing audio...
Detected language: English


 99%|█████████▉| 33728/34088 [02:02<00:01, 274.80frames/s]


✅ Transcription complete! Length: 6950 characters
✨ Summarizing text...
   Summarizing chunk 1/8...
   Summarizing chunk 2/8...
   Summarizing chunk 3/8...
   Summarizing chunk 4/8...
   Summarizing chunk 5/8...
   Summarizing chunk 6/8...
   Summarizing chunk 7/8...


Your max_length is set to 150, but your input_length is only 55. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=27)


   Summarizing chunk 8/8...
✅ Summary generated!
📄 Creating study guide...
✅ Study guide saved: study_guide.docx
🤖 Setting up chatbot...
   Created 16 text chunks
   Creating embeddings...
   Building vector store...
   Loading QA model...
✅ Chatbot ready!
